Urban Data Science & Smart Cities <br>
URSP688Y Spring 2026<br>
Instructor: Chester Harvey <br>
Urban Studies & Planning <br>
National Center for Smart Growth <br>
University of Maryland

# Demo 5 - More Tables & Debugging

- More Tables
    - Drop duplicate rows
    - Count rows withing groups
    - Concatenate tables
    - Join columns from another table
- Errors and Debugging

## Data Wrangling with Tables

In [1]:
# Import Pandas package for working with tables
import pandas as pd

In [2]:
# Load raw eviction data
eviction_cases_df = pd.read_csv('District_Court_of_Maryland_Eviction_Case_Data_MG_PG.csv')

In [3]:
# Investivate the dataframe
eviction_cases_df.head()

,Unnamed: 0,Event Date,Event Type,Event Comment,County,Location,Tenant City,Tenant State,Tenant ZIP Code,Case Type,Case Number,Evicted Date,Event Year,Eviction Year
0,535,01/03/2023,Warrant of Restitution - Return of Service - E...,NaN,Montgomery,Rockville,Silver Spring,MD,20910.0,Failure to Pay Rent,D-061-LT-22-004107,12/08/2022,2023,2022.0
1,536,01/03/2023,Warrant of Restitution - Return of Service - E...,NaN,Montgomery,Rockville,Silver Spring,MD,20910.0,Failure to Pay Rent,D-061-LT-22-000755,12/08/2022,2023,2022.0
2,545,01/03/2023,Warrant of Restitution - Return of Service - E...,NaN,Montgomery,Rockville,Bethesda,MD,20814.0,Failure to Pay Rent,D-061-LT-22-000816,12/07/2022,2023,2022.0
3,555,01/03/2023,Warrant of Restitution - Return of Service - E...,NaN,Montgomery,Rockville,Silver Spring,MD,20902.0,Failure to Pay Rent,D-061-LT-22-006362,12/08/2022,2023,2022.0
4,568,01/03/2023,Warrant of Restitution - Return of Service - E...,NaN,Montgomery,Rockville,Rockville,MD,20850.0,Failure to Pay Rent,D-061-LT-22-004268,12/07/2022,2023,2022.0


In [4]:
eviction_cases_df['Event Type'].value_counts()

Event Type
Petition - For Warrant of Restitution Filed               96046
Warrant of Restitution - Return of Service - Cancelled    44912
Warrant of Restitution - Return of Service - Evicted      15159
Warrant of Restitution - Return of Service - Expired       4322
petition - For Warrant of Restitution Filed                  43
Name: count, dtype: int64

In [5]:
# Convert data in date columns to datetime objects so they can be sorted properly

def convert_column_to_datetime(df, column):
    """
    Converts a column in a dataframe to datetime objects

    Args:
        df (Pandas DataFrame): input dataframe
        column (string): column in input dataframe to be converted to datetime

    Returns:
        Pandas DataFrame: copy of input dataframe with the converted datetime column
    """
    df = df.copy()
    df[column] = pd.to_datetime(df[column])
    return df

for date_column in ['Event Date', 'Evicted Date']:
    eviction_cases_df = convert_column_to_datetime(eviction_cases_df, date_column)

In [6]:
# Investigate data types of columns
eviction_cases_df.dtypes

Unnamed: 0                  int64
Event Date         datetime64[us]
Event Type                    str
Event Comment                 str
County                        str
Location                      str
Tenant City                   str
Tenant State                  str
Tenant ZIP Code           float64
Case Type                     str
Case Number                   str
Evicted Date       datetime64[us]
Event Year                  int64
Eviction Year             float64
dtype: object

How many unique cases are there?

In [9]:
len(eviction_cases_df)

160482

In [13]:
# .drop_duplicates() method
eviction_cases_df = eviction_cases_df.drop_duplicates('Case Number')

In [11]:
# help(eviction_cases_df.drop_duplicates)

How many unique cases per zip code?

In [14]:
# .groupby() method
eviction_cases_df.groupby('Tenant ZIP Code').size()

Tenant ZIP Code
19802.0    1
20001.0    2
20010.0    1
20011.0    2
20074.0    1
          ..
28014.0    2
29094.0    1
29747.0    1
29748.0    2
55403.0    1
Length: 147, dtype: int64

In [15]:
eviction_cases_df.head()

,Unnamed: 0,Event Date,Event Type,Event Comment,County,Location,Tenant City,Tenant State,Tenant ZIP Code,Case Type,Case Number,Evicted Date,Event Year,Eviction Year
0,535,2023-01-03,Warrant of Restitution - Return of Service - E...,NaN,Montgomery,Rockville,Silver Spring,MD,20910.0,Failure to Pay Rent,D-061-LT-22-004107,2022-12-08,2023,2022.0
1,536,2023-01-03,Warrant of Restitution - Return of Service - E...,NaN,Montgomery,Rockville,Silver Spring,MD,20910.0,Failure to Pay Rent,D-061-LT-22-000755,2022-12-08,2023,2022.0
2,545,2023-01-03,Warrant of Restitution - Return of Service - E...,NaN,Montgomery,Rockville,Bethesda,MD,20814.0,Failure to Pay Rent,D-061-LT-22-000816,2022-12-07,2023,2022.0
3,555,2023-01-03,Warrant of Restitution - Return of Service - E...,NaN,Montgomery,Rockville,Silver Spring,MD,20902.0,Failure to Pay Rent,D-061-LT-22-006362,2022-12-08,2023,2022.0
4,568,2023-01-03,Warrant of Restitution - Return of Service - E...,NaN,Montgomery,Rockville,Rockville,MD,20850.0,Failure to Pay Rent,D-061-LT-22-004268,2022-12-07,2023,2022.0


In [16]:
eviction_cases_df.groupby('Tenant ZIP Code')['Event Year'].max()

Tenant ZIP Code
19802.0    2023
20001.0    2024
20010.0    2024
20011.0    2024
20074.0    2024
           ... 
28014.0    2025
29094.0    2025
29747.0    2025
29748.0    2025
55403.0    2023
Name: Event Year, Length: 147, dtype: int64

In [18]:
eviction_cases_df.groupby('Tenant ZIP Code').agg({'Event Year': 'max','Event Date': 'count'})

,Event Year,Event Date
Tenant ZIP Code,,
19802.0,2023,1
20001.0,2024,2
20010.0,2024,1
20011.0,2024,2
20074.0,2024,1
...,...,...
28014.0,2025,2
29094.0,2025,1
29747.0,2025,1


Which zip codes have the most unique cases *per person*?

Let's join data from [CensusReporter](https://censusreporter.org/).

### Combining/Merging/Joining Tables

Combining information from multiple tables into a single table is one of the most useful data wrangling operations.

There are lots of different ways to join tables, but two basic types are:

1. Joining column with a shared key, which outputs a table that is wider than either input.
2. Concatenating rows with shared column names, which outputs a table that is longer than either input.

#### Joining columns based on a key

![joining columns with a shared key](https://rforhr.com/horizontal_join.png)


#### Concatenating rows with the same column names
![joining rows with shared column names](https://rforhr.com/vertical_join.png)

In [20]:
# Load census reporter data, ignoring the row with data for the whole county (first row under the header)
acs_mg_df = pd.read_csv('acs2024_5yr_B01003_mg.csv')
acs_pg_df = pd.read_csv('acs2024_5yr_B01003_pg.csv')


In [21]:
acs_mg_df.head()

,geoid,name,B01003001,"B01003001, Error"
0,05000US24031,"Montgomery County, MD",1065949,NaN
1,86000US20705,20705,28713,1967.0
2,86000US20707,20707,37278,1391.0
3,86000US20777,20777,3221,541.0
4,86000US20812,20812,276,95.0


In [33]:
# Can we write a function that loads a census reporter csv and skips the second row?
def load_census_reporter_csv(path):
    df = pd.read_csv(path)
    # take all rows except the 
    df = df.iloc[1:]
    # reset the index to start at 0
    df = df.reset_index(drop=True)
    # rename columns for readability
    df = df.rename(columns={'B01003_001E': 'population'})
    # drop unnecessary columns
    df = df.drop['geoid','population']
    return df

acs_mg_df = load_census_reporter_csv('acs2024_5yr_B01003_mg.csv')
acs_pg_df = load_census_reporter_csv('acs2024_5yr_B01003_pg.csv')


TypeError: 'method' object is not subscriptable

In [31]:
# Combine into a single dataframe

# pd.concat() function
pd.contact([acs_mg_df, acs_pg_df])

AttributeError: module 'pandas' has no attribute 'contact'

In [ ]:
# Rename columns with readable names
column_map = {
    'name':'census_zip', 
    'B01003001':'population', 
    'B01003001, Error':'population_error'
}

In [ ]:
# Make sure zip codes are stored as strings in both the eviction and census dataframes

# .astype() method

In [ ]:
# Clean up the dataframe

# .fillna() method

# .drop() method

In [ ]:
# Calculate evictions per population

## Errors and debugging

Errors are frustrating and inevitable. Even professional programmers spend much of their time debugging.

Luckily, there are good tools and techniques for making debugging a little easier.

Despite these, you will probably nearly tear your hair out with some frequency, especially as a beginner. It will get better with time.

There are two types of errors in programming: logic and syntax. They both result in your program not achieving its goal, but the first may not be as easily detectable because the code may still run.

### Logic errors
These are issues with how you have approached or executed your problem. If your code runs but produces nonsensical results, there is probably a logic error. However, your erroneous code might also produce logical but *wrong* results; you might never notice until the problem has rippled downstream. It's best to address this proactively by planning your code well so it's less likely to be illogical, and writing readable code that can be easily reviewed.

Here's a logic error. Can you find it? (Hint: the issue is syntactical, but it's still a logic error because the code works without throwing an error.)

In [ ]:
def check_adult(age, cutoff=18):
    if age > cutoff:
        adult = False
    else:
        adult = True
    return adult

check_adult(20)

### Syntax errors
These are more obvious because your code will simply fail. There are lots of tools for figuring out where and why.

Error messages are usually the starting place for debugging a syntax error.

In [ ]:
def check_adult(age, cutoff=18):
    if age < cutoff:
        adult = False
    else:
        adult = True
    return adult

check_adult('20')

### Debugging
We can also use an "interactive debugger" to help diagnose our problem by stepping through the code one line at a time.

The debugger allows you to set "breakpoints" where the code will stop running temporarily, a table that shows the values of variables at that time, and buttons to step through the code.

In [ ]:
def check_adult(age, cutoff=18):
    if age < cutoff:
        adult = False
    else:
        adult = True
    return adult

check_adult(10)